In [21]:
import os
import sys
os.environ['PYTHONPATH'] = os.path.expanduser('~/mhbai')
sys.path.append(os.path.expanduser('~/mhbai'))

In [7]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("jinaai/jina-embeddings-v5-text-nano-text-matching", trust_remote_code=True)

[transformers] Unrecognized keys in `rope_parameters` for 'rope_type'='default': {'factor'}
/home/gatterle/mhbai/venv/lib/python3.12/site-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)


Loading weights:   0%|          | 0/110 [00:00<?, ?it/s]

[transformers] Unrecognized keys in `rope_parameters` for 'rope_type'='default': {'factor'}
[transformers] Unrecognized keys in `rope_parameters` for 'rope_type'='default': {'factor'}


In [8]:
from database import database as db

result = db.select(
    table="unia.modules_ai_extracted",
    columns=["id", "module_code", "contents", "goals", "raw_module_id"],
    conditions={"module_code": "LMZ-1713"},
    type_of_answer=db.ANSWER_TYPE.SINGLE_ANSWER
)

if result.is_error:
    raise UserWarning("Error fetching data:", result.error)

data = result.data
contents = data["contents"]
goals = data["goals"]
print("Contents:", contents)
print("Goals:", goals)

Contents: ['Geschichte der populären Musik', 'Historische, musikalische und soziokulturelle Aspekte der Populären Musik', 'Musikmedien, Live- und Studiotechnik']
Goals: ['berufsfeldspezifische Kenntnisse im Hinblick auf historische, musikalische und soziokulturelle Bedingungen und Entwicklungen der Populären Musik']


In [15]:
from database import database as db

result = db.select(
    table="unia.modules",
    columns=["id", "module_code", "content", "goals"],
    conditions={"module_code": "LMZ-1713"},
    type_of_answer=db.ANSWER_TYPE.SINGLE_ANSWER
)

if result.is_error:
    raise UserWarning("Error fetching data:", result.error)

data = result.data
contents = data["content"]
goals = data["goals"]
print("Contents:", contents)
print("Goals:", goals)

Contents: Historische, musikalische und soziokulturelle Aspekte der Populären Musik.
Goals: 


In [30]:
# German-specific, trained on STS tasks
model = SentenceTransformer("sentence-transformers/distiluse-base-multilingual-cased-v2")
# or
# model = SentenceTransformer("sentence-transformers/xlm-r-distilroberta-base")

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

In [31]:
sentences = [
    "Diese Modul enth\u00e4lt eine Reihe von Veranstaltungen im Fach Geographie, die f\u00fcr Studierende und Interessierte des Fachs angeboten werden um die Auseinandersetzung mit fachlichen Fragen auf einem wissenschaftlichen Niveau zu f\u00f6rdern. Die Teilnahme ist freiwillig. Genaue Angaben zu den Themen beziehungsweise einzelnen Vortr\u00e4gen innerhalb der Angebote entnehmen Sie bitte den Ank\u00fcndigungen unter Aktuelles auf der Institutshomepage oder den ausgeh\u00e4ngten Plakaten.",
    "\n\nGrundlegende geographiedidaktische Konzepte und Bewertung geographischer Unterrichtsinhalte;\n                        \n                        Grundlegender Zugang zur Rolle von Unterrichtsmethoden und Medien bei der Planung des Geographieunterrichts;\n                        \n                        Grundlegende schulart\u00fcbergreifende und schulartspezifische Planung von Unterricht",
]
print(sentences[0])
print(sentences[1])

embeddings = model.encode(sentences)

similarities = model.similarity(embeddings, embeddings)
print(similarities)

Diese Modul enthält eine Reihe von Veranstaltungen im Fach Geographie, die für Studierende und Interessierte des Fachs angeboten werden um die Auseinandersetzung mit fachlichen Fragen auf einem wissenschaftlichen Niveau zu fördern. Die Teilnahme ist freiwillig. Genaue Angaben zu den Themen beziehungsweise einzelnen Vorträgen innerhalb der Angebote entnehmen Sie bitte den Ankündigungen unter Aktuelles auf der Institutshomepage oder den ausgehängten Plakaten.


Grundlegende geographiedidaktische Konzepte und Bewertung geographischer Unterrichtsinhalte;
                        
                        Grundlegender Zugang zur Rolle von Unterrichtsmethoden und Medien bei der Planung des Geographieunterrichts;
                        
                        Grundlegende schulartübergreifende und schulartspezifische Planung von Unterricht
tensor([[1.0000, 0.3133],
        [0.3133, 1.0000]])


In [42]:
result = db.select(
    table="unia.modules_ai_extracted",
    columns=["id", "title", "module_code", "ects", "lecturer", "contents", "goals", "requirements", "expense", "success_requirements", "weekly_hours", "recommended_semester", "exams", "module_parts", "raw_module_id"],
    type_of_answer=db.ANSWER_TYPE.LIST_ANSWER
)
data = result.data
data = [i for i in data if i["contents"] is not None and i["goals"] is not None and i["contents"] != [] and i["goals"] != []]
print(len(data))
print(data[0])
lmz = [i for i in data if i["module_code"].startswith("LMZ-")]

23689
{'id': 10951, 'title': 'Modul LMZ-1536: Wahlmodul Konzertprojekt II', 'module_code': 'LMZ-1536', 'ects': 1, 'lecturer': '', 'contents': ['Durchführung eines speziellen Konzertprojektes im Rahmen der „Klingenden Bibliothek“, in einer Schule, einem Krankenhaus, einem Seniorenheim etc.'], 'goals': ['Musikalische Gestaltung eines Konzertes, Auftrittspraxis'], 'requirements': ['Bestehen der Modulprüfung'], 'expense': [{'hours': 30, 'activity': 'Durchführung eines speziellen Konzertprojektes'}], 'success_requirements': ['Bestehen der Modulprüfung'], 'weekly_hours': 0, 'recommended_semester': 1, 'exams': [{'duration': None, 'exam_info': ['Beteiligungsnachweis, unbenotet'], 'exam_type': 'Keine'}], 'module_parts': [{'title': 'Konzertprojekt II', 'language': 'Deutsch', 'teaching_methods': ['Übung']}], 'raw_module_id': 2193683}


In [47]:
from collections import defaultdict
d = defaultdict(int)
for i in lmz:
    d[i["module_code"]] += 1
print(d)

for i in lmz:
    if i["module_code"] == "LMZ-1536":
        print(i["contents"])

defaultdict(<class 'int'>, {'LMZ-1536': 24, 'LMZ-1549': 53, 'LMZ-1504': 7, 'LMZ-1538': 26, 'LMZ-1529': 19, 'LMZ-0905': 28, 'LMZ-1556': 48, 'LMZ-1527': 58, 'LMZ-1555': 47, 'LMZ-1802': 28, 'LMZ-1550': 56, 'LMZ-1593': 35, 'LMZ-1530': 33, 'LMZ-1552': 47, 'LMZ-1510': 59, 'LMZ-1502': 20, 'LMZ-1808': 29, 'LMZ-1503': 33, 'LMZ-1594': 34, 'LMZ-1809': 12, 'LMZ-1551': 53, 'LMZ-1710': 12, 'LMZ-1537': 24, 'LMZ-1597': 14, 'LMZ-1595': 34, 'LMZ-1509': 52, 'LMZ-0904': 2, 'LMZ-1229': 36, 'LMZ-1541': 7, 'LMZ-1521': 25, 'LMZ-1540': 59, 'LMZ-1105': 46, 'LMZ-1506': 7, 'LMZ-1505': 7, 'LMZ-1546': 62, 'LMZ-0709': 16, 'LMZ-1228': 36, 'LMZ-1811': 13, 'LMZ-1106': 47, 'LMZ-1542': 7, 'LMZ-1553': 33, 'LMZ-1515': 19, 'LMZ-1568': 2, 'LMZ-1533': 8, 'LMZ-1564': 28, 'LMZ-1543': 7, 'LMZ-1508': 7, 'LMZ-1720': 20, 'LMZ-1712': 26, 'LMZ-1201': 11, 'LMZ-1501': 24, 'LMZ-1711': 25, 'LMZ-1534': 7, 'LMZ-1104': 33, 'LMZ-1519': 45, 'LMZ-1558': 2, 'LMZ-1102': 36, 'LMZ-1101': 43, 'LMZ-1585': 19, 'LMZ-1569': 9, 'LMZ-0706': 15, 'LMZ-1203

In [17]:
from sentence_transformers import SentenceTransformer
import numpy as np

model = SentenceTransformer("sentence-transformers/paraphrase-multilingual-mpnet-base-v2")

sent1 = "Diese Modul enthält eine Reihe von Veranstaltungen... Die Teilnahme ist freiwillig."
sent2 = "Grundlegende geographiedidaktische Konzepte und Bewertung..."

# Extract features
emb1 = model.encode(sent1)  # 768-dim vector
emb2 = model.encode(sent2)  # 768-dim vector

# Now you can:
# 1. Calculate similarity
from sklearn.metrics.pairwise import cosine_similarity
sim = cosine_similarity([emb1], [emb2])[0][0]

# 2. Cluster similar sentences
from sklearn.cluster import KMeans
embeddings = np.array([emb1, emb2, ...])
clusters = KMeans(n_clusters=3).fit_predict(embeddings)

# 3. Add custom logic (keywords, modifiers)
if "freiwillig" in sent1 and "freiwillig" not in sent2:
    # Reduce similarity to account for pragmatic difference
    adjusted_sim = sim * 0.7

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

ValueError: setting an array element with a sequence. The requested array has an inhomogeneous shape after 1 dimensions. The detected shape was (3,) + inhomogeneous part.

In [16]:
from transformers import AutoTokenizer, AutoModelForMaskedLM

tokenizer = AutoTokenizer.from_pretrained("distilbert/distilbert-base-german-cased")
model = AutoModelForMaskedLM.from_pretrained("distilbert/distilbert-base-german-cased")

config.json:   0%|          | 0.00/464 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/270M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]